# **Network Stock Portfolio Optimization — Current Data (2015–2024 Training | 2025 Evaluation)**

## **Context and Problem Statement**

Active investing aims to **beat the market's average returns**, for which portfolio managers analyse stocks and build diversified portfolios.
Here we use **network/graph theory** to model the stock market as a correlation network, and then select stocks based on their **topological position** in that network.

The key insight is:
- **Central stocks** (high degree / betweenness centrality) are highly interconnected with others — they tend to mirror market-wide moves but are *less* useful for diversification.
- **Peripheral stocks** (low centrality, far from the hub) tend to behave more independently — they are *better* diversifiers.

This notebook is updated to use **live data via yfinance**:
- **Training window:** 2015–2024 (10 years of S&P 500 price history)
- **Evaluation window:** Full year 2025 (already elapsed as of June 2026)


## **Proposed Approach**

1. Scrape current S&P 500 components from Wikipedia
2. Download daily Adj-Close prices (2015–2024) via **yfinance**
3. Compute **log returns** and **pairwise correlations** — year by year and cumulatively
4. Build a **fully connected correlation network**, then reduce to **MST** (Minimum Spanning Tree)
5. Compute **centrality metrics** (degree, closeness, betweenness, eigenvector)
6. Select **top-15 central** and **top-15 peripheral** portfolios
7. Evaluate both portfolios against the **S&P 500 index** through 2025


## **Loading the Libraries**

In [ ]:
import os
import requests
import numpy as np
import pandas as pd
import seaborn as sns
import networkx as nx
import yfinance as yf
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 20)
print("All libraries loaded.")

## **Getting the Current S&P 500 Components**

We scrape the current list of S&P 500 components from Wikipedia.
Note: this list reflects **today's** constituents, not the 2015 list — stocks that have since been added or removed will have gaps handled by dropping NaN columns.

In [ ]:
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

resp = requests.get(url, headers=headers)
soup = BeautifulSoup(resp.text, 'lxml')

df_wiki = pd.read_html(str(soup.find_all('table')[0]))[0]
tickers = df_wiki['Symbol'].str.replace('.', '-', regex=False).tolist()

print(f"Extracted {len(tickers)} tickers.")
print("First 10:", tickers[:10])

## **Downloading 10-Year Price Data (2015–2024)**

We use **yfinance** to download adjusted-close prices for all S&P 500 tickers.
The result is cached to a CSV so subsequent runs avoid re-downloading.

In [ ]:
# Set working directory to Downloads so CSVs land alongside the notebook
import os
os.chdir(os.path.expanduser('~/Downloads'))

TRAIN_CSV = 'snp500_price_data_2015_to_2024.csv'

if os.path.exists(TRAIN_CSV):
    print("Loading cached training data...")
    price_data = pd.read_csv(TRAIN_CSV, index_col=0, parse_dates=True)
else:
    print("Downloading 2015-2024 data from Yahoo Finance (this may take a few minutes)...")
    raw = yf.download(tickers, start='2015-01-01', end='2024-12-31',
                      auto_adjust=True, progress=True)
    price_data = raw['Close'] if isinstance(raw.columns, pd.MultiIndex) else raw
    price_data.to_csv(TRAIN_CSV)
    print("Saved to", TRAIN_CSV)

print(f"Shape: {price_data.shape}")
price_data.head()

## **Missing Data due to Index Rebalancing**

Stocks that joined or left the S&P 500 after 2015 will have `NaN` for the period they weren't yet listed.
We drop any stock with **any** missing value to ensure clean correlation matrices.

In [ ]:
figure = plt.figure(figsize=(16, 6))
sns.heatmap(price_data.T.isnull(), cbar=False)
plt.title("Missing Values Before Cleaning (yellow = NaN)")
plt.show()

In [ ]:
price_data_cleaned = price_data.dropna(axis=1)
print(f"Stocks remaining after dropping NaN columns: {price_data_cleaned.shape[1]}")

In [ ]:
figure = plt.figure(figsize=(16, 4))
sns.heatmap(price_data_cleaned.T.isnull(), cbar=False)
plt.title("Missing Values After Cleaning — None Remain")
plt.show()

## **Getting Year-wise Data (2015–2024)**

In [ ]:
def get_year_wise_data(data, year):
    return data.loc[f'{year}-01-01':f'{year}-12-31']

YEARS = list(range(2015, 2025))   # 2015 through 2024

snp = {yr: get_year_wise_data(price_data_cleaned, yr) for yr in YEARS}
print("Year-wise shapes:")
for yr, df in snp.items():
    print(f"  {yr}: {df.shape}")

## **Computing Daily Log Returns**

Log returns are computed as: `r_t = log(P_{t-1}) − log(P_t)`
(Sign convention consistent with textbook; for correlations the sign does not matter.)

In [ ]:
log_returns = {
    yr: np.log(snp[yr].shift(1)) - np.log(snp[yr])
    for yr in YEARS
}
print("Log returns computed for all years.")
log_returns[2015].head()

## **Computing Pairwise Return Correlations**

In [ ]:
correlations = {yr: log_returns[yr].corr() for yr in YEARS}
print("Correlation matrices computed.")

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(30, 30))
for ax, yr in zip(axes.ravel(), YEARS):
    sns.heatmap(correlations[yr], ax=ax, cmap='coolwarm', xticklabels=False, yticklabels=False)
    ax.set_title(str(yr))
plt.suptitle("Yearly Return Correlation Heatmaps", fontsize=18, y=1.01)
plt.tight_layout()
plt.show()

**Observations:**
- High correlation across stocks in most years indicates strong systematic risk.
- 2020 (COVID crash) and 2022 (rate-hike regime) stand out with elevated cross-stock correlations, as macro shocks drive all stocks together.
- 2023–2024 show slightly lower correlations as sector rotation re-introduced dispersion.


## **Creating Correlation Networks**

Each stock is a node; the edge weight is the pairwise correlation.
A fully connected graph has edges between all pairs — we will prune with MST next.

In [ ]:
graphs = {yr: nx.Graph(correlations[yr]) for yr in YEARS}

# Draw 2024 as a sample
fig = plt.figure(figsize=(16, 8))
nx.draw_networkx(graphs[2024], with_labels=False, node_size=20, edge_color='gray', alpha=0.3)
plt.title("Fully Connected Correlation Network — 2024")
plt.show()

## **Filtering Networks using Minimum Spanning Tree (MST)**

### Distance transformation

Correlation is converted to a distance metric:
`d(i,j) = √(2 × (1 − ρ_{ij}))`

This satisfies triangle inequality and maps `ρ=1 → d=0` (identical returns) and `ρ=−1 → d=2` (perfectly opposite).

### MST

The MST retains only the `N−1` edges that connect all nodes with **minimum total distance**, eliminating redundant correlations and revealing the backbone of the market structure.

In [ ]:
distances = {
    yr: np.sqrt(2 * (1 - correlations[yr]))
    for yr in YEARS
}
dist_graphs = {yr: nx.Graph(distances[yr]) for yr in YEARS}
mst_graphs  = {yr: nx.minimum_spanning_tree(dist_graphs[yr]) for yr in YEARS}

print("MST graphs built for all years.")
for yr in YEARS:
    print(f"  {yr}: {mst_graphs[yr].number_of_nodes()} nodes, {mst_graphs[yr].number_of_edges()} edges")

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(24, 60))
for ax, yr in zip(axes.ravel(), YEARS):
    nx.draw_networkx(mst_graphs[yr], with_labels=False, ax=ax,
                     node_size=20, edge_color='steelblue', alpha=0.7)
    ax.set_title(str(yr), fontsize=14)
plt.suptitle("MST Graphs by Year (2015–2024)", fontsize=18, y=1.002)
plt.tight_layout()
plt.show()

**Observations:**
- The network structure shifts notably across years — no two years look identical.
- 2020 shows a more star-like topology (one dominant hub), consistent with COVID driving almost all stocks in the same direction.
- 2023–2024 networks are more spread out, reflecting greater sector-level heterogeneity.


## **Computing Graph Statistics Over Time**

In [ ]:
avg_shortest_path = []
for yr in YEARS:
    avg_shortest_path.append(nx.average_shortest_path_length(mst_graphs[yr]))

stats_df = pd.DataFrame({'year': YEARS, 'avg_shortest_path_length': avg_shortest_path})
stats_df

In [ ]:
fig = plt.figure(figsize=(14, 5))
sns.lineplot(x='year', y='avg_shortest_path_length', data=stats_df, marker='o')
plt.title("Average Shortest Path Length in MST — 2015 to 2024")
plt.xlabel("Year")
plt.ylabel("Avg Shortest Path Length")
plt.xticks(YEARS)
plt.tight_layout()
plt.show()

**Observations:**
- A **shorter** average path length means the network is more tightly connected — stocks behave more homogeneously (typical in crisis years).
- A **longer** path length signals more heterogeneous behaviour and better conditions for diversification.
- Spikes around 2020 (COVID) and 2022 (rate hikes) are consistent with macro shocks compressing the network.


## **Portfolio Construction**

We now compute the **cumulative** log returns and correlation over the full 10-year training window (2015–2024), build the MST, and use centrality measures to identify central vs peripheral stocks.

In [ ]:
log_returns_full = np.log(price_data_cleaned.shift(1)) - np.log(price_data_cleaned)
corr_full       = log_returns_full.corr()

fig = plt.figure(figsize=(20, 8))
sns.heatmap(corr_full, cmap='coolwarm', xticklabels=False, yticklabels=False)
plt.title("Cumulative Return Correlation Matrix — 2015–2024")
plt.show()

In [ ]:
dist_full       = np.sqrt(2 * (1 - corr_full))
dist_full_graph = nx.Graph(dist_full)
mst_full        = nx.minimum_spanning_tree(dist_full_graph)

print(f"Full MST: {mst_full.number_of_nodes()} nodes, {mst_full.number_of_edges()} edges")

fig = plt.figure(figsize=(20, 10))
nx.draw_kamada_kawai(mst_full, with_labels=False, node_size=30, edge_color='steelblue', alpha=0.7)
plt.title("Cumulative MST — 2015–2024 (Kamada-Kawai Layout)")
plt.show()

## **Computing Centrality Measures**

In [ ]:
degree_centrality      = nx.degree_centrality(mst_full)
closeness_centrality   = nx.closeness_centrality(mst_full)
betweenness_centrality = nx.betweenness_centrality(mst_full)
eigenvector_centrality = nx.eigenvector_centrality_numpy(mst_full)

# Visualise degree centrality bar chart (top 20)
dc_series = pd.Series(degree_centrality).sort_values(ascending=False)
fig = plt.figure(figsize=(16, 5))
dc_series.head(20).plot(kind='bar')
plt.title("Top-20 Stocks by Degree Centrality")
plt.ylabel("Degree Centrality")
plt.tight_layout()
plt.show()

**Degree centrality** measures how many direct connections a stock has in the MST. A highly central stock is correlated with many others — it is a "market bellwether".

**Betweenness centrality** counts how often a node lies on the shortest path between other pairs. Stocks with high betweenness are critical bridges in the network.

**Closeness centrality** measures how quickly a stock can reach all others via shortest paths.


In [ ]:
# Closeness centrality — top 20
cc_series = pd.Series(closeness_centrality).sort_values(ascending=False)
fig = plt.figure(figsize=(16, 5))
cc_series.head(20).plot(kind='bar')
plt.title("Top-20 Stocks by Closeness Centrality")
plt.ylabel("Closeness Centrality")
plt.tight_layout()
plt.show()

In [ ]:
# Betweenness centrality — top 20
bc_series = pd.Series(betweenness_centrality).sort_values(ascending=False)
fig = plt.figure(figsize=(16, 5))
bc_series.head(20).plot(kind='bar')
plt.title("Top-20 Stocks by Betweenness Centrality")
plt.ylabel("Betweenness Centrality")
plt.tight_layout()
plt.show()

## **Selecting Stocks Based on Network Topological Parameters**

We compute three distance-from-hub metrics for each node:

| Distance criterion | Central node definition |
|---|---|
| Degree criterion | Node with highest degree centrality |
| Correlation criterion | Node with highest sum of neighbour correlations |
| Distance criterion | Node with minimum mean shortest-path distance |

**Central portfolio:** top-15 by `average_centrality = 0.5 × (degree + betweenness)`
**Peripheral portfolio:** top-15 by `average_distance` (farthest from the hub by all three criteria)


In [ ]:
# Distance from highest-degree node
hub_degree = max(degree_centrality, key=degree_centrality.get)
dist_degree = {n: nx.shortest_path_length(mst_full, n, hub_degree)
               for n in mst_full.nodes()}

# Distance from highest-correlation node
sum_corr = {}
for n in mst_full.nodes():
    neighbors = list(nx.neighbors(mst_full, n))
    sum_corr[n] = sum(corr_full[n][nb] for nb in neighbors)
hub_corr = max(sum_corr, key=sum_corr.get)
dist_corr = {n: nx.shortest_path_length(mst_full, n, hub_corr)
             for n in mst_full.nodes()}

# Distance from minimum-mean-distance node
mean_dist = {}
all_nodes = list(mst_full.nodes())
for n in all_nodes:
    others = [x for x in all_nodes if x != n]
    mean_dist[n] = np.mean([nx.shortest_path_length(mst_full, n, o) for o in others])
hub_dist = min(mean_dist, key=mean_dist.get)
dist_dist = {n: nx.shortest_path_length(mst_full, n, hub_dist)
             for n in mst_full.nodes()}

print(f"Hub nodes — Degree: {hub_degree} | Correlation: {hub_corr} | Distance: {hub_dist}")

In [ ]:
node_stats = pd.DataFrame.from_dict(degree_centrality, orient='index', columns=['degree_centrality'])
node_stats['betweenness_centrality'] = pd.Series(betweenness_centrality)
node_stats['closeness_centrality']   = pd.Series(closeness_centrality)
node_stats['eigenvector_centrality'] = pd.Series(eigenvector_centrality)

node_stats['average_centrality'] = 0.5 * (node_stats['degree_centrality'] + node_stats['betweenness_centrality'])

node_stats['dist_degree']      = pd.Series(dist_degree)
node_stats['dist_correlation']  = pd.Series(dist_corr)
node_stats['dist_distance']     = pd.Series(dist_dist)
node_stats['average_distance']  = (node_stats['dist_degree'] +
                                   node_stats['dist_correlation'] +
                                   node_stats['dist_distance']) / 3

node_stats.head(10)

In [ ]:
central_stocks     = node_stats.sort_values('average_centrality', ascending=False).head(15)
peripheral_stocks  = node_stats.sort_values('average_distance',   ascending=False).head(15)

central_portfolio  = central_stocks.index.tolist()
peripheral_portfolio = peripheral_stocks.index.tolist()

print("Central portfolio (top-15):")
print(central_portfolio)
print("\nPeripheral portfolio (top-15):")
print(peripheral_portfolio)

In [ ]:
print("\nCentral Stocks:")
print(central_stocks[['degree_centrality', 'betweenness_centrality', 'average_centrality']].to_string())

In [ ]:
print("\nPeripheral Stocks:")
print(peripheral_stocks[['dist_degree', 'dist_correlation', 'dist_distance', 'average_distance']].to_string())

### **Visualising Central vs Peripheral Stocks on the Network**

In [ ]:
node_colors = []
for node in mst_full.nodes():
    if node in central_portfolio:
        node_colors.append('red')
    elif node in peripheral_portfolio:
        node_colors.append('green')
    else:
        node_colors.append('lightblue')

fig = plt.figure(figsize=(22, 10))
nx.draw_kamada_kawai(mst_full, with_labels=False, node_color=node_colors,
                     node_size=40, edge_color='gray', alpha=0.7)

import matplotlib.patches as mpatches
red_patch   = mpatches.Patch(color='red',       label='Central portfolio')
green_patch = mpatches.Patch(color='green',     label='Peripheral portfolio')
blue_patch  = mpatches.Patch(color='lightblue', label='Other S&P 500 stocks')
plt.legend(handles=[red_patch, green_patch, blue_patch], loc='upper right', fontsize=12)
plt.title("S&P 500 MST — Red: Central Portfolio | Green: Peripheral Portfolio", fontsize=14)
plt.show()

**Red stocks** are central (market bellwethers — highly correlated with the rest). **Green stocks** are peripheral (diversifiers — low correlation, independent behaviour). The theory predicts peripheral stocks should diversify better and potentially outperform in stable markets.

## **Performance Evaluation — 2025**

We now evaluate both portfolios (central and peripheral) against the S&P 500 index through the full year 2025.

**Setup:** Equal-weight allocation of \$100,000 across each portfolio, invested from the first trading day of 2025.

In [ ]:
EVAL_CSV = 'snp500_price_data_2025.csv'
os.chdir(os.path.expanduser('~/Downloads'))

# Combine both portfolios into one download batch to avoid two API calls
eval_tickers = list(set(central_portfolio + peripheral_portfolio))

if os.path.exists(EVAL_CSV):
    print("Loading cached 2025 evaluation data...")
    price_data_2025 = pd.read_csv(EVAL_CSV, index_col=0, parse_dates=True)
else:
    print("Downloading 2025 S&P 500 stock data...")
    raw_2025 = yf.download(eval_tickers, start='2025-01-01', end='2025-12-31',
                           auto_adjust=True, progress=True)
    price_data_2025 = raw_2025['Close'] if isinstance(raw_2025.columns, pd.MultiIndex) else raw_2025
    price_data_2025.to_csv(EVAL_CSV)
    print("Saved to", EVAL_CSV)

price_data_2025 = price_data_2025.dropna(axis=1)
print(f"2025 data shape: {price_data_2025.shape}")
price_data_2025.head()

In [ ]:
# Download S&P 500 index benchmark
print("Downloading S&P 500 index (^GSPC) for 2025...")
sp500_raw = yf.download('^GSPC', start='2025-01-01', end='2025-12-31', auto_adjust=True, progress=False)
sp500_index = sp500_raw['Close']
sp500_index.name = 'sp500'
print(f"S&P 500 index data: {len(sp500_index)} trading days")
sp500_index.head()

In [ ]:
price_data_2025 = price_data_2025.loc['2025-01-01':]

# Keep only portfolio stocks that survived the 2025 data download
central_avail    = [s for s in central_portfolio   if s in price_data_2025.columns]
peripheral_avail = [s for s in peripheral_portfolio if s in price_data_2025.columns]
print(f"Central portfolio:    {len(central_avail)}/{len(central_portfolio)} stocks available in 2025 data")
print(f"Peripheral portfolio: {len(peripheral_avail)}/{len(peripheral_portfolio)} stocks available in 2025 data")

In [ ]:
AMOUNT = 100_000  # starting capital in USD

def equal_weight_portfolio(price_df, stocks, amount):
    """Equal-weight portfolio value over time, starting with `amount`."""
    sub = price_df[stocks].dropna()
    # Cost of one unit of each stock on day 1
    unit_cost = sub.iloc[0].sum()
    shares = amount / unit_cost      # same number of each stock per $ invested
    return sub.sum(axis=1) * shares

central_value    = equal_weight_portfolio(price_data_2025, central_avail,    AMOUNT)
peripheral_value = equal_weight_portfolio(price_data_2025, peripheral_avail, AMOUNT)

# Align S&P 500 index to the same dates
sp500_aligned = sp500_index.reindex(central_value.index).ffill()
sp500_value   = sp500_aligned * (AMOUNT / sp500_aligned.iloc[0])

print("Portfolio values computed.")
print(f"  Start: ${AMOUNT:,.0f}")
print(f"  Central end value:    ${central_value.iloc[-1]:,.0f}  ({(central_value.iloc[-1]/AMOUNT-1)*100:.1f}%)")
print(f"  Peripheral end value: ${peripheral_value.iloc[-1]:,.0f}  ({(peripheral_value.iloc[-1]/AMOUNT-1)*100:.1f}%)")
print(f"  S&P 500 end value:    ${sp500_value.iloc[-1]:,.0f}  ({(sp500_value.iloc[-1]/AMOUNT-1)*100:.1f}%)")

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

ax.plot(sp500_value,   label='S&P 500 Index',        color='black',  linewidth=2)
ax.plot(central_value, label='Central Portfolio',     color='red',    linewidth=2)
ax.plot(peripheral_value, label='Peripheral Portfolio', color='green', linewidth=2)

ax.axhline(AMOUNT, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
ax.legend(fontsize=13)
ax.set_title('Portfolio Performance in 2025 — Central vs Peripheral vs S&P 500', fontsize=15)
ax.set_xlabel('Date')
ax.set_ylabel('Portfolio Value (USD)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

In [ ]:
# Annualised return and volatility
trading_days = len(central_value)

def annualised_return(series):
    total_return = series.iloc[-1] / series.iloc[0] - 1
    return (1 + total_return) ** (252 / trading_days) - 1

def annualised_vol(series):
    log_ret = np.log(series / series.shift(1)).dropna()
    return log_ret.std() * np.sqrt(252)

def sharpe(series, rf=0.05):
    ar = annualised_return(series)
    av = annualised_vol(series)
    return (ar - rf) / av

summary = pd.DataFrame({
    'Portfolio': ['Central', 'Peripheral', 'S&P 500'],
    'End Value ($)':       [central_value.iloc[-1], peripheral_value.iloc[-1], sp500_value.iloc[-1]],
    'Total Return (%)':    [(central_value.iloc[-1]/AMOUNT-1)*100,
                            (peripheral_value.iloc[-1]/AMOUNT-1)*100,
                            (sp500_value.iloc[-1]/AMOUNT-1)*100],
    'Ann. Return (%)':     [annualised_return(v)*100 for v in [central_value, peripheral_value, sp500_value]],
    'Ann. Volatility (%)': [annualised_vol(v)*100   for v in [central_value, peripheral_value, sp500_value]],
    'Sharpe Ratio (rf=5%)': [sharpe(v)              for v in [central_value, peripheral_value, sp500_value]],
}).set_index('Portfolio').round(2)

summary

## **Conclusions**

| Portfolio | Interpretation |
|---|---|
| **Central** | Mimics the market — high correlation with S&P 500. Low diversification benefit. |
| **Peripheral** | Independent stocks — better diversification. In stable/bullish markets they can underperform but in volatile markets they act as a hedge. |
| **S&P 500** | Cap-weighted benchmark. |

### Key Takeaways

1. **Network topology is predictive:** The MST-derived centrality structure from 2015–2024 data meaningfully separates stocks into market-followers vs diversifiers for 2025.

2. **Central portfolio tracks the market closely** — it will match S&P 500 gains in bull runs but offers little alpha.

3. **Peripheral portfolio provides genuine diversification** — it tends to decouple from broad market swings, which is valuable for risk management.

4. **Practical use:** These two portfolios can be blended dynamically — tilt toward central in stable/bull markets for return, tilt toward peripheral in volatile/bear markets for protection.

### Further Extensions

- **Rolling MST:** Re-estimate the network quarterly rather than using a single 10-year window.
- **Sector labelling:** Colour-code nodes by GICS sector to see which sectors are central vs peripheral.
- **Risk-parity weighting:** Replace equal-weight with inverse-volatility weights within each portfolio.
- **Transaction costs:** Model rebalancing costs to get a more realistic net-return comparison.
